# Module 4 — VIX term structure data check

Before building `analysis.py`, confirm what historical IV-proxy data is actually
available for free via yfinance, across the horizons we want for the VRP
decomposition (Bakshi & Kapadia style structural premium vs. forecast error).

`^VIX1Y` is excluded — Yahoo only mirrors its most recent single day, no
historical series, so it's unusable here. If a 1Y bucket is wanted later,
it would need CBOE's own historical CSV download as a separate manual step.

In [2]:
import sys
sys.path.append('..')

import logging
logging.basicConfig(level=logging.INFO)

import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

INFO:numexpr.utils:NumExpr defaulting to 4 threads.


## Fetch VIX term structure tickers

`^VIX9D` (9 calendar days), `^VIX` (30 calendar days), `^VIX3M` (~91 calendar
days / 63 trading days), `^VIX6M` (~182 calendar days / 126 trading days).

In [3]:
VIX_TICKERS = ["^VIX9D", "^VIX", "^VIX3M", "^VIX6M"]

vix_series = {}
for ticker in VIX_TICKERS:
    raw = yf.download(ticker, period="max", auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    close = raw["Close"].dropna().sort_index()
    vix_series[ticker] = close
    print(f"{ticker}: {len(close)} rows, {close.index[0].date()} to {close.index[-1].date()}")

^VIX9D: 3885 rows, 2011-01-03 to 2026-06-16
^VIX: 9182 rows, 1990-01-02 to 2026-06-16
^VIX3M: 5010 rows, 2006-07-17 to 2026-06-16
^VIX6M: 4642 rows, 2008-01-02 to 2026-06-16


## Common overlap window

All four horizons need to be available on the same date for a given row of the
decomposition — find the intersection of their date ranges.

In [4]:
vix_df = pd.DataFrame(vix_series)
vix_df_common = vix_df.dropna()

print(f"Full (outer) range: {vix_df.index[0].date()} to {vix_df.index[-1].date()}, {len(vix_df)} rows")
print(f"Common (all 4 present) range: {vix_df_common.index[0].date()} to {vix_df_common.index[-1].date()}, {len(vix_df_common)} rows")
vix_df_common.tail()

Full (outer) range: 1990-01-02 to 2026-06-16, 9182 rows
Common (all 4 present) range: 2011-01-03 to 2026-06-16, 3885 rows


,^VIX9D,^VIX,^VIX3M,^VIX6M
Date,,,,
2026-06-09,22.139999,19.870001,21.309999,22.969999
2026-06-10,25.670000,22.219999,22.889999,24.129999
2026-06-11,20.660000,19.440001,21.420000,23.139999
2026-06-12,17.260000,17.680000,20.510000,22.590000
2026-06-16,15.770000,16.410000,19.530001,21.870001


## Quick visual sanity check

Term structure should generally fan out sensibly (VIX9D most volatile/noisy,
VIX6M smoothest), with all four co-moving in level.

In [5]:
fig = go.Figure()
for ticker in VIX_TICKERS:
    fig.add_trace(go.Scatter(
        x=vix_df_common.index,
        y=vix_df_common[ticker],
        mode="lines",
        name=ticker,
        line=dict(width=1.2),
    ))

fig.update_layout(
    title="VIX term structure — common history",
    xaxis_title="Date",
    yaxis_title="Level",
    template="plotly_dark",
    hovermode="x unified",
)
fig.show()

---

## Step 2 — GARCH(1,1) walk-forward term structure

### The model

GARCH(1,1) models time-varying conditional variance as:

$$\sigma^2_t = \omega + \alpha \, r^2_{t-1} + \beta \, \sigma^2_{t-1}$$

| Parameter | Role |
|-----------|------|
| $\omega > 0$ | Baseline variance floor |
| $\alpha$ (ARCH term) | Weight on yesterday's squared shock — captures volatility clustering |
| $\beta$ (GARCH term) | Persistence — how slowly variance decays back to long-run mean |

The long-run (unconditional) variance is $\bar{\sigma}^2 = \omega / (1 - \alpha - \beta)$, which is finite only when $\alpha + \beta < 1$ (stationarity condition).

### Walk-forward discipline

At each date $t$, the model is **re-fit on the trailing 252 trading days only**, and the forecast is written at position $t$. No future returns ever enter the estimate — the output is exactly what a real-time system would have produced.

### Multi-step forecasting and mean-reversion

The $k$-step-ahead variance forecast recurses as:

$$E_t[\sigma^2_{t+k}] = \bar{\sigma}^2 + (\alpha+\beta)^{k-1}\bigl(E_t[\sigma^2_{t+1}] - \bar{\sigma}^2\bigr)$$

Because $\alpha + \beta < 1$, every additional step damps the deviation from $\bar{\sigma}^2$ by factor $(\alpha+\beta)$. Short-horizon forecasts track current vol conditions closely; long-horizon forecasts converge toward $\bar{\sigma}^2$.

### Aggregating to $N$-day annualised vol

For each horizon $N \in \{21, 63, 126\}$ trading days:

$$\hat{\sigma}^{(N)}_t = \sqrt{\frac{252}{N} \sum_{k=1}^{N} E_t[\sigma^2_{t+k}]}$$

This averages the daily variance path over $N$ days, then annualises. In `walk_forward_garch_term_structure`, a single `res.forecast(horizon=max\_N)` call supplies the full variance path; cumulative sums give each horizon without re-fitting.

In [6]:
from src.realised_vol import (
    fetch_price_history,
    compute_log_returns,
    walk_forward_garch_term_structure,
    DEFAULT_TERM_HORIZONS,
    DEFAULT_TRAIN_WINDOW,
    TRADING_DAYS_PER_YEAR,
)

prices = fetch_price_history("SPY", period="5y")
returns = compute_log_returns(prices)
print(f"SPY: {len(returns)} daily returns, {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Forecasts will start after {DEFAULT_TRAIN_WINDOW}-day warmup → "
      f"{len(returns) - DEFAULT_TRAIN_WINDOW} forecast rows")

INFO:src.realised_vol:Fetching price history for SPY, period=5y
INFO:src.realised_vol:Fetched 1255 daily closes for SPY (2021-06-17 to 2026-06-16)
INFO:src.realised_vol:Computed 1254 log returns (2021-06-18 to 2026-06-16)


SPY: 1254 daily returns, 2021-06-18 → 2026-06-16
Forecasts will start after 252-day warmup → 1002 forecast rows


In [7]:
# ~2–3 min for 5y of SPY (one GARCH fit per trading day)
term_df = walk_forward_garch_term_structure(
    returns,
    horizons=DEFAULT_TERM_HORIZONS,
    train_window=DEFAULT_TRAIN_WINDOW,
)
term_df.tail()

INFO:src.realised_vol:Starting walk-forward GARCH term structure: 1002 steps, train_window=252, horizons=[21, 63, 126]
INFO:src.realised_vol:Walk-forward term structure progress: 10%
INFO:src.realised_vol:Walk-forward term structure progress: 20%
INFO:src.realised_vol:Walk-forward term structure progress: 30%
INFO:src.realised_vol:Walk-forward term structure progress: 40%
INFO:src.realised_vol:Walk-forward term structure progress: 50%
INFO:src.realised_vol:Walk-forward term structure progress: 60%
INFO:src.realised_vol:Walk-forward term structure progress: 70%
INFO:src.realised_vol:Walk-forward term structure progress: 80%
INFO:src.realised_vol:Walk-forward term structure progress: 90%
INFO:src.realised_vol:Walk-forward term structure progress: 100%
INFO:src.realised_vol:Walk-forward term structure complete: 1002/1002 fully valid rows


,garch_forecast_21d,garch_forecast_63d,garch_forecast_126d
Date,,,
2026-06-10,0.144361,0.147577,0.152274
2026-06-11,0.144620,0.147688,0.152174
2026-06-12,0.145733,0.148798,0.153282
2026-06-15,0.144683,0.147581,0.151824
2026-06-16,0.145416,0.148221,0.152332


In [8]:
print("=== Shape & coverage ===")
print(f"Shape: {term_df.shape}")
for col in term_df.columns:
    n_valid = term_df[col].notna().sum()
    print(f"  {col}: {n_valid} non-NaN ({n_valid / len(term_df):.1%})")

print(f"\n=== NaN prefix — first {DEFAULT_TRAIN_WINDOW} rows all NaN ===")
nan_block = term_df.iloc[:DEFAULT_TRAIN_WINDOW]
print(f"  All NaN: {nan_block.isnull().all(axis=None)}")
print(f"  First forecast date: {term_df.dropna().index[0].date()}")

print("\n=== Most-recent single-date term structure ===")
latest = term_df.dropna().iloc[-1]
for col, val in latest.items():
    print(f"  {col}: {val:.2%}")

print("\n=== Cumulative variance monotone with horizon ===")
# cum_var(h) = ann_vol^2 * h / 252 must grow because we sum non-negative daily variances
valid = term_df.dropna()
for h1, h2 in zip(DEFAULT_TERM_HORIZONS[:-1], DEFAULT_TERM_HORIZONS[1:]):
    cv1 = valid[f"garch_forecast_{h1}d"] ** 2 * h1 / TRADING_DAYS_PER_YEAR
    cv2 = valid[f"garch_forecast_{h2}d"] ** 2 * h2 / TRADING_DAYS_PER_YEAR
    print(f"  cumvar({h2}d) >= cumvar({h1}d) on all dates: {(cv2 >= cv1 - 1e-12).all()}")

print("\n=== Level plausibility (expect 5%–80% for SPY) ===")
desc = valid.describe()
for col in desc.columns:
    print(f"  {col}: min={desc[col]['min']:.2%}, median={desc[col]['50%']:.2%}, max={desc[col]['max']:.2%}")

=== Shape & coverage ===
Shape: (1254, 3)
  garch_forecast_21d: 1002 non-NaN (79.9%)
  garch_forecast_63d: 1002 non-NaN (79.9%)
  garch_forecast_126d: 1002 non-NaN (79.9%)

=== NaN prefix — first 252 rows all NaN ===
  All NaN: True
  First forecast date: 2022-06-17

=== Most-recent single-date term structure ===
  garch_forecast_21d: 14.54%
  garch_forecast_63d: 14.82%
  garch_forecast_126d: 15.23%

=== Cumulative variance monotone with horizon ===
  cumvar(63d) >= cumvar(21d) on all dates: True
  cumvar(126d) >= cumvar(63d) on all dates: True

=== Level plausibility (expect 5%–80% for SPY) ===
  garch_forecast_21d: min=8.64%, median=13.70%, max=77.39%
  garch_forecast_63d: min=8.12%, median=13.58%, max=75.52%
  garch_forecast_126d: min=7.44%, median=13.00%, max=73.04%


In [9]:
fig2 = go.Figure()

_colors = {
    "garch_forecast_21d":  "#EF553B",
    "garch_forecast_63d":  "#00CC96",
    "garch_forecast_126d": "#636EFA",
}
_labels = {
    "garch_forecast_21d":  "GARCH 21d (~1mo)",
    "garch_forecast_63d":  "GARCH 63d (~3mo)",
    "garch_forecast_126d": "GARCH 126d (~6mo)",
}

for col in term_df.columns:
    fig2.add_trace(go.Scatter(
        x=term_df.index,
        y=term_df[col] * 100,
        mode="lines",
        name=_labels[col],
        line=dict(color=_colors[col], width=1.5),
    ))

fig2.update_layout(
    title="GARCH(1,1) walk-forward term structure — SPY (annualised vol %)",
    xaxis_title="Date",
    yaxis_title="Annualised vol forecast (%)",
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig2.show()

In [ ]:
from src.analysis import VIX_HORIZON_MAP, _tz_naive

vix_safe  = _tz_naive(vix_df_common)
term_safe = _tz_naive(term_df.dropna())

shared_dates = term_safe.index.intersection(vix_safe.index)
latest_date  = shared_dates[-1]
print(f"Snapshot date: {latest_date.date()}\n")

rows = []
for ticker, h in VIX_HORIZON_MAP.items():
    iv_pct    = vix_safe.loc[latest_date, ticker]
    garch_pct = term_safe.loc[latest_date, f"garch_forecast_{h}d"] * 100
    rows.append({
        "Horizon":            f"~{h}d",
        "VIX ticker":         ticker,
        "IV (%)":             round(iv_pct, 2),
        "GARCH (%)":          round(garch_pct, 2),
        "Gap IV−GARCH (pp)":  round(iv_pct - garch_pct, 2),
    })

pd.DataFrame(rows).set_index("Horizon")

## Results interpretation

**Term structure shape (14.5% → 14.8% → 15.2%)**  
The GARCH forecast increases only slightly with horizon (~0.7 pp across 21→126d). Current SPY vol is sitting close to the model's long-run mean, so there's almost no mean-reversion pull in either direction. The term structure is essentially flat.

**The VIX term structure is far steeper**  
See the VRP preview table below. VIX slopes from 16.4% to 21.9% (+5.5 pp) while GARCH barely moves (+0.7 pp). The market is pricing substantially more uncertainty at 3–6 months than GARCH would justify — this excess slope is the structural premium component that `analysis.py` is designed to isolate.

**Gap widens sharply with horizon**  
The IV−GARCH gap is small at 21d (~2 pp, roughly in line with historical averages) but large at 126d (~6–7 pp). This is consistent with the Bakshi & Kapadia finding that the structural premium grows with horizon, not just with current vol level.

**Max GARCH vol of ~77% (21d)**  
The highest 21d forecast in the sample (see plausibility stats above) is driven by the post-tariff shock in April 2025, when SPY fell ~10% in two days. GARCH responds quickly to large squared returns, so the conditional variance spikes sharply then decays. This is expected behaviour, not a data issue.

---

**Note on VIX9D**  
VIX9D measures ~9 calendar days = ~6 trading days. The GARCH term structure starts at 21d, so there's no clean horizon match for it. It also sets the binding constraint on the common VIX history start (2011 instead of 2006). For those two reasons we drop it: the analysis uses VIX (21d), VIX3M (63d), VIX6M (126d) only, with common data available from 2008. The `VIX_TICKERS` and `GARCH_HORIZONS` constants in `src/analysis.py` reflect this.

---

## Step 3 — Forward realised volatility

### Definition

At date $t$, the $N$-day **forward realised variance** uses the $N$ daily log-returns that immediately follow $t$:

$$\text{RV}^2(t,\, t+N) \;=\; \frac{252}{N} \sum_{k=1}^{N} r_{t+k}^2$$

Taking the square root gives forward realised vol in the same units as VIX — annualised, in decimal form.

**No lookahead:** $r_{t+1}, \ldots, r_{t+N}$ are all strictly future relative to the signal date $t$. The last $N$ dates in the sample cannot form a complete window and yield `NaN`. This is not a data issue — those rows are dropped automatically when aligning with IV and GARCH in Step 4.

This is the volatility that investors at date $t$ were exposed to but *could not observe*. Comparing it to $\text{IV}(t)$ gives the total gap; inserting the GARCH forecast in between gives the decomposition.

### Implementation trick

`squared.rolling(N).sum()` at position $i$ accumulates $\sum_{k=i-N+1}^{i} r_k^2$ — a **backward** window.  
`.shift(-N)` moves the series back $N$ rows so that position $i$ now holds what was at $i+N$, i.e. $\sum_{k=i+1}^{i+N} r_k^2$ — the **forward** window we need.

The daily mean return is treated as zero, which is standard for daily log-returns (the mean contributes $\sim\!0.3\%$ of total variance at these horizons and is negligible).

In [ ]:
from src.analysis import compute_forward_realised_vol, GARCH_HORIZONS

forward_rv_df = compute_forward_realised_vol(returns, GARCH_HORIZONS)

print(f"Shape: {forward_rv_df.shape}")
print(f"\nValid range per horizon:")
for col in forward_rv_df.columns:
    valid = forward_rv_df[col].dropna()
    n_nan = forward_rv_df[col].isna().sum()
    print(f"  {col}: {valid.index[0].date()} → {valid.index[-1].date()},  "
          f"{len(valid)} valid rows,  {n_nan} NaN")
forward_rv_df.tail(8)

In [17]:
print("=== NaN tail counts — expect exactly N NaN at end of each column ===")
for col in forward_rv_df.columns:
    n        = int(col.split("_")[-1].rstrip("d"))
    n_nan    = forward_rv_df[col].isna().sum()
    tail_nan = forward_rv_df[col].iloc[-n:].isna().sum()
    pre_nan  = forward_rv_df[col].iloc[:-n].isna().sum()
    print(f"  {col}: total NaN = {n_nan}  "
          f"(tail {n} rows → {tail_nan} NaN, earlier rows → {pre_nan} NaN)")

print("\n=== Level plausibility (expect 5%–80% for SPY) ===")
desc = forward_rv_df.dropna().describe()
for col in desc.columns:
    print(f"  {col}: min={desc[col]['min']:.2%}, median={desc[col]['50%']:.2%}, max={desc[col]['max']:.2%}")

print("\n=== Spot-check: correlation of 21d forward RV vs VIX ===")
fwd_21 = forward_rv_df["forward_rv_21d"].dropna()
iv_21  = vix_safe["^VIX"] / 100
joint  = pd.concat([iv_21.rename("iv_vix"), fwd_21.rename("fwd_rv_21d")], axis=1).dropna()
corr   = joint.corr().iloc[0, 1]
print(f"  Pearson r (IV vs forward RV, 21d, n={len(joint)}): {corr:.3f}")
print(f"  Expect ~0.5–0.8 — VIX over-predicts on average but co-moves with realised vol")

=== NaN tail counts — expect exactly N NaN at end of each column ===
  forward_rv_21d: total NaN = 21  (tail 21 rows → 21 NaN, earlier rows → 0 NaN)
  forward_rv_63d: total NaN = 63  (tail 63 rows → 63 NaN, earlier rows → 0 NaN)
  forward_rv_126d: total NaN = 126  (tail 126 rows → 126 NaN, earlier rows → 0 NaN)

=== Level plausibility (expect 5%–80% for SPY) ===
  forward_rv_21d: min=6.15%, median=13.56%, max=51.28%
  forward_rv_63d: min=8.64%, median=13.26%, max=33.40%
  forward_rv_126d: min=10.61%, median=13.46%, max=26.64%

=== Spot-check: correlation of 21d forward RV vs VIX ===
  Pearson r (IV vs forward RV, 21d, n=1233): 0.619
  Expect ~0.5–0.8 — VIX over-predicts on average but co-moves with realised vol


---

## Step 4 — VRP decomposition

### The three components

At each date $t$ and horizon $N$, the total gap between implied and realised vol decomposes into two non-overlapping parts:

| Component | Formula | What it captures |
|-----------|---------|-----------------|
| **Total gap** | $\text{IV}(t) - \text{RV}(t, t+N)$ | Everything the market priced above actual future vol |
| **Structural premium** | $\text{IV}(t) - \hat{\sigma}^{(N)}_t$ | What the market charged above the GARCH-rational forecast |
| **Forecast error** | $\hat{\sigma}^{(N)}_t - \text{RV}(t, t+N)$ | How far the GARCH model itself was from future vol |

The accounting identity holds exactly by construction — the GARCH forecast cancels:

$$\underbrace{\text{IV}(t) - \text{RV}(t,t+N)}_{\text{total gap}} \;=\; \underbrace{\text{IV}(t) - \hat{\sigma}^{(N)}_t}_{\text{structural premium}} \;+\; \underbrace{\hat{\sigma}^{(N)}_t - \text{RV}(t,t+N)}_{\text{forecast error}}$$

**Interpretation:**
- A **positive structural premium** means the market demanded compensation for variance risk beyond what a rational model would forecast — the classical risk premium of Bakshi & Kapadia (2003).
- A **positive forecast error** means GARCH over-estimated future vol; common in low-vol regimes where the model is slow to decay toward the long-run mean.
- The structural premium isolates the component of the total gap that *cannot* be attributed to model inaccuracy, and is therefore driven by risk aversion / demand for crash insurance.

### Horizon matching

| VIX ticker | Calendar days | Trading-day horizon $N$ | GARCH column |
|-----------|--------------|------------------------|-------------|
| `^VIX` | ~30 | 21 | `garch_forecast_21d` |
| `^VIX3M` | ~91 | 63 | `garch_forecast_63d` |
| `^VIX6M` | ~182 | 126 | `garch_forecast_126d` |

VIX is quoted in annualised vol (%), so we divide by 100 before differencing with GARCH or RV values, which are in decimal form.

In [ ]:
from src.analysis import decompose_vrp

In [ ]:
forward_rv_safe = _tz_naive(forward_rv_df)

vrp_df = decompose_vrp(
    vix_df        = vix_safe,
    garch_df      = term_safe,
    forward_rv_df = forward_rv_safe,
)

print(f"Shape: {vrp_df.shape}")
print(f"Columns: {list(vrp_df.columns)}")
print(f"\nRows per horizon:")
for h, grp in vrp_df.groupby("horizon"):
    print(f"  {h}d: {len(grp)} rows   ({grp['date'].min().date()} → {grp['date'].max().date()})")

vrp_df.head(6)

In [20]:
print("=== Identity: total_gap == structural_premium + forecast_error ===")
residual = (
    vrp_df["total_gap"] - vrp_df["structural_premium"] - vrp_df["forecast_error"]
).abs()
print(f"  Max |residual|: {residual.max():.2e}  (expect < 1e-14)")

print("\n=== Mean components by horizon (annualised vol, pp) ===")
summary = (
    vrp_df
    .groupby("horizon")[["total_gap", "structural_premium", "forecast_error"]]
    .mean()
    .mul(100)
    .round(2)
)
summary.columns = ["Mean total gap (pp)", "Mean struct. premium (pp)", "Mean forecast error (pp)"]
print(summary.to_string())

print("\n=== Fraction of dates where structural premium is positive ===")
for h, grp in vrp_df.groupby("horizon"):
    pos = (grp["structural_premium"] > 0).mean()
    print(f"  {h:3d}d:  {pos:.1%} of dates  IV > GARCH forecast")

=== Identity: total_gap == structural_premium + forecast_error ===
  Max |residual|: 5.55e-17  (expect < 1e-14)

=== Mean components by horizon (annualised vol, pp) ===
         Mean total gap (pp)  Mean struct. premium (pp)  Mean forecast error (pp)
horizon                                                                          
21                      3.73                       2.59                      1.14
63                      5.06                       3.91                      1.15
126                     6.56                       5.44                      1.12

=== Fraction of dates where structural premium is positive ===
   21d:  82.9% of dates  IV > GARCH forecast
   63d:  88.6% of dates  IV > GARCH forecast
  126d:  93.5% of dates  IV > GARCH forecast


In [21]:
import plotly.subplots as sp

_h_info = [(21, "^VIX", "~21d"), (63, "^VIX3M", "~63d"), (126, "^VIX6M", "~126d")]
_comp_colors = {
    "total_gap":           "#AB63FA",
    "structural_premium":  "#EF553B",
    "forecast_error":      "#00CC96",
}
_comp_labels = {
    "total_gap":           "Total gap  IV − RV",
    "structural_premium":  "Structural premium  IV − GARCH",
    "forecast_error":      "Forecast error  GARCH − RV",
}

fig3 = sp.make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Horizon {label} ({ticker})" for _, ticker, label in _h_info],
    vertical_spacing=0.07,
)
for row_i, (h, _, _label) in enumerate(_h_info, start=1):
    sub = vrp_df[vrp_df["horizon"] == h].sort_values("date")
    for comp, color in _comp_colors.items():
        fig3.add_trace(
            go.Scatter(
                x=sub["date"],
                y=sub[comp] * 100,
                mode="lines",
                name=_comp_labels[comp],
                line=dict(color=color, width=1.3),
                showlegend=(row_i == 1),
            ),
            row=row_i, col=1,
        )
    fig3.add_hline(y=0, line_dash="dot", line_color="grey", line_width=0.8, row=row_i, col=1)

fig3.update_layout(
    title="VRP decomposition — total gap, structural premium, forecast error (annualised vol, pp)",
    height=780,
    template="plotly_dark",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1),
)
fig3.update_yaxes(title_text="Vol (pp)")
fig3.show()